# RAG_APP: Crave Assist, the CraveCart help assistant

Builds on the class `RAG_APP.ipynb` (LangChain + Chroma + Groq + Gradio) and adds:

- **Follow-up rewriting:** each question plus the chat history becomes one standalone English search query, so follow-ups and Hindi/Hinglish questions retrieve the right passages.
- **Citations:** answers mark the passages they use as [1], [2]…, and the app shows them.
- **Exact order maths:** the LLM calls Python tools for cancellation fees and refunds instead of guessing.
- **Streaming, voice (Whisper), Hindi/Hinglish replies, and a Gemini backup** when Groq is rate-limited.

**Keys:** in Colab, add `GROQ_API_KEY` (and optionally `GOOGLE_API_KEY`) under 🔑 Secrets. Locally, put them in `.env`.

## Setup

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
SAVE_DB_TO_DRIVE = False  # True: keep the vector DB in Google Drive, like the class notebook

if IN_COLAB:
    if not Path("ai-engineering-projects").exists():
        !git clone --depth 1 https://github.com/C0derANANT/ai-engineering-projects.git
    %cd ai-engineering-projects/03-Food-Delivery-RAG
    %pip install -q -r requirements.txt
    if SAVE_DB_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["VECTOR_DB_DIR"] = "/content/drive/MyDrive/ai/cravecart_vector_db"
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project folder so `rag` can be imported

sys.path.insert(0, str(Path.cwd()))
print("Project folder:", Path.cwd())

if IN_COLAB:
    from google.colab import userdata
    for name in ("GROQ_API_KEY", "GOOGLE_API_KEY"):
        try:
            os.environ[name] = userdata.get(name) or ""
        except Exception:
            print(f"{name} not found in Colab secrets" + (" (optional)" if name == "GOOGLE_API_KEY" else ""))

## 1. Connect the vector DB, demo orders, tools and models
Builds the database if it doesn't exist yet (or if the documents changed).

In [ ]:
from rag.service import build_service
from rag.llm import get_models

service = build_service()
models = get_models(service.settings)
print(f"Vector DB: {service.db_info.chunks} chunks · demo orders: {', '.join(service.orders.ids())}")
print("Models:", models.names if models else "none (add GROQ_API_KEY)")

## 2. Query rewriting
The raw message is turned into a standalone English search query, and its language is detected.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

history = [HumanMessage(content="I want to cancel order 1042"), AIMessage(content="You'd get ₹299.79 back.")]
for question, past in [("Mere order 1045 mein Mutton Korma nahi aaya, kitna refund milega?", []),
                       ("what if the restaurant had already started cooking?", history)]:
    query, language = service.assistant.rewrite_query(models, past, question)
    print(f"{question}\n  -> query: {query!r} · language: {language}\n")

## 3. Tools: exact cancellation and refund maths
These are the functions the LLM calls. The numbers match the worked examples in the policy documents.

In [ ]:
import json
tools = {t.name: t for t in service.assistant.tools}

quote = tools["cancellation_quote"].invoke({"order_id": "1042"})
print("Cancel #1042 ->", quote["stage"], "| fee", quote["cancellation_fee"], "| refund", quote["refund_total"])

refund = tools["item_refund_quote"].invoke({"order_id": "1045", "items": [{"name": "Mutton Korma"}], "issue": "missing"})
print("Missing Mutton Korma on #1045 ->", refund["refund_total"])
print(json.dumps(refund["breakdown"], indent=2, ensure_ascii=False))

## 4. Ask questions
`ask()` runs the whole pipeline (rewrite → retrieve → answer with tools) and shows the sources.

In [ ]:
from IPython.display import Markdown, display
from rag.ui import sources_markdown

chat_history = []

def ask(question):
    reply = None
    for reply in service.assistant.respond(question, chat_history):
        pass  # the app streams these updates; here we just keep the last one
    chat_history.extend([{"role": "user", "content": question}, {"role": "assistant", "content": reply.text}])
    display(Markdown(f"**Q: {question}**\n\n{reply.text}\n\n---\n{sources_markdown(reply)}"))

ask("I want to cancel order 1042. How much will I get back?")

In [ ]:
ask("And what if the restaurant had already started cooking?")  # a follow-up

In [ ]:
ask("मेरा ऑर्डर 1044 बहुत लेट है, क्या मुझे कुछ मिलेगा?")

## 5. Launch the Gradio app
In Colab this prints a public `gradio.live` link.

In [ ]:
import app  # builds the UI from app.py

if os.environ.get("SKIP_LAUNCH") != "1":
    app.launch(share=IN_COLAB, inbrowser=not IN_COLAB)